<a href="https://colab.research.google.com/github/RogerVin1/PA3_Deep_Learning/blob/main/PA3_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%cd /content
!git clone https://github.com/RogerVin1/PA3_Deep_Learning.git
%cd /content/PA3_Deep_Learning

/content
Cloning into 'PA3_Deep_Learning'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/PA3_Deep_Learning


In [3]:
%cd /content/PA3_Deep_Learning
!mkdir -p pa3 scripts results && touch results/.gitkeep
!pwd && ls -a

/content/PA3_Deep_Learning
/content/PA3_Deep_Learning
.  ..  .git  .gitignore  pa3  README.md  requirements.txt  results  scripts


In [4]:
from pathlib import Path

Path(".gitignore").write_text('''# Dados (415 MB, não versionar)
data/
pa3_dados/
*.zip

# Checkpoints de treino (o modelo final vai em weights/ ou em link externo)
checkpoints/
*.pt
*.pth
!weights/modelo_final.pt

# Python / Jupyter
__pycache__/
*.pyc
.ipynb_checkpoints/
.pytest_cache/
.venv/
venv/

# Sistema
.DS_Store
''')

Path("requirements.txt").write_text('''# No Colab, torch e torchvision já vêm instalados: só instale o resto.
torch>=2.2
torchvision>=0.17
scikit-learn>=1.3
pandas>=2.0
numpy>=1.24
matplotlib>=3.7
Pillow>=10.0
pytest>=7.4
''')

Path("pa3/__init__.py").write_text('''"""PA3 — Poucos rótulos: fauna em câmeras trap (ENA24) com dados não rotulados."""

# Referência do monitor: ResNet-18 (ImageNet) com todos os rótulos do train,
# época escolhida no val, F1 macro no test_public. É o "teto" da Parte 2.
F1_TETO_MONITOR = 0.686

# Limite do enunciado para o modelo entregue.
MAX_PARAMS = 12_000_000

# Orçamentos de rótulos por classe (rotulos_1/5/20.csv). O padrão é 5.
ORCAMENTOS = (1, 5, 20)
ORCAMENTO_PADRAO = 5
''')

447

In [5]:
!pip -q install gdown
!mkdir -p /content/data && cd /content/data && gdown 1g7FGqtXOk_ziNKBwQAk8BxjPOaPCnbui -O pa3_dados.zip

Downloading...
From (original): https://drive.google.com/uc?id=1g7FGqtXOk_ziNKBwQAk8BxjPOaPCnbui
From (redirected): https://drive.google.com/uc?id=1g7FGqtXOk_ziNKBwQAk8BxjPOaPCnbui&confirm=t&uuid=269a002a-50d2-434d-b744-cd4a7e698927
To: /content/data/pa3_dados.zip
100% 436M/436M [00:08<00:00, 50.1MB/s]


In [6]:
!cd /content/data && unzip -q -o pa3_dados.zip && ls -la
!du -sh /content/data
!find /content/data -maxdepth 2 | head -40
!find /content/data -type f \( -name "*.jpg" -o -name "*.jpeg" -o -name "*.png" \) | wc -l

total 425308
drwxr-xr-x 3 root root      4096 Nov 15 22:04 .
drwxr-xr-x 1 root root      4096 Nov 15 22:03 ..
drwxr-xr-x 4 root root      4096 Nov 15 22:05 pa3_dados
-rw-r--r-- 1 root root 435500297 Oct 31 19:09 pa3_dados.zip
845M	/content/data
/content/data
/content/data/pa3_dados
/content/data/pa3_dados/labels
/content/data/pa3_dados/images
/content/data/pa3_dados/LEIAME.md
/content/data/pa3_dados/classes.csv
/content/data/pa3_dados.zip
7518


In [8]:
%%writefile pa3/data.py
"""Leitura do pacote de dados do PA3 (ENA24-detection, 21 classes).

O enunciado descreve o conteúdo do pacote (train, val, test_public,
rotulos_1/5/20.csv, classes.csv), mas não o layout exato de pastas nem os nomes
das colunas dos CSVs. Por isso o carregador *descobre* o layout em vez de
assumi-lo: procura o classes.csv, indexa todas as imagens por split e detecta
as colunas de id e de rótulo. Se algo não bater, o erro diz o que foi
encontrado; rode `python scripts/inspect_data.py` e ajuste as constantes abaixo.
"""
from __future__ import annotations

import warnings
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_DIRS = ("train", "val", "test_public", "test_private")

# Nomes de coluna aceitos (minúsculas). Ajuste aqui se o pacote usar outros.
ID_COLS = ("image_id", "id", "image", "img", "filename", "file_name", "file", "path", "arquivo")
LABEL_COLS = ("label", "labels", "class", "classe", "class_name", "species", "especie", "target", "category")
_CLASS_HEADER_WORDS = {"id", "class_id", "idx", "index", "class", "classe", "class_name",
                       "name", "nome", "label", "species", "especie"}

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


# --------------------------------------------------------------------------- #
# Descoberta do layout
# --------------------------------------------------------------------------- #
def find_root(path: str | Path) -> Path:
    """Devolve a pasta que contém classes.csv (procura dentro de `path`)."""
    p = Path(path).expanduser()
    if not p.exists():
        raise FileNotFoundError(
            f"Pasta de dados não encontrada: {p}\n"
            "Baixe e descompacte pa3_dados.zip e passe --data CAMINHO "
            "(ou defina a variável de ambiente PA3_DATA)."
        )
    hits = sorted(p.rglob("classes.csv"), key=lambda q: len(q.parts))
    if not hits:
        raise FileNotFoundError(
            f"Não achei classes.csv dentro de {p}. Conteúdo: "
            f"{sorted(x.name for x in p.iterdir())[:20]}"
        )
    return hits[0].parent


def read_classes(root: Path) -> list[str]:
    """Nomes das classes, na ordem dos índices 0..K-1."""
    path = Path(root) / "classes.csv"
    raw = pd.read_csv(path, header=None, dtype=str, keep_default_na=False)
    first = [str(c).strip().lower() for c in raw.iloc[0]]
    df = pd.read_csv(path, dtype=str, keep_default_na=False) if any(
        c in _CLASS_HEADER_WORDS for c in first) else raw
    df = df.apply(lambda col: col.str.strip())
    if df.shape[1] == 1:
        return df.iloc[:, 0].tolist()
    id_col = next((c for c in df.columns if df[c].str.fullmatch(r"\d+").all()), None)
    name_col = next(c for c in df.columns if c != id_col)
    if id_col is not None:
        df = df.assign(_i=df[id_col].astype(int)).sort_values("_i")
    return df[name_col].tolist()


class ImageIndex:
    """Índice stem -> caminho de todas as imagens abaixo de `root`, por split."""

    def __init__(self, root: Path):
        self.by_split: dict[str, dict[str, Path]] = {s: {} for s in SPLIT_DIRS}
        self.any: dict[str, Path] = {}
        for p in sorted(Path(root).rglob("*")):
            if p.suffix.lower() not in IMG_EXT:
                continue
            parent_parts = p.relative_to(root).parts[:-1]
            for s in SPLIT_DIRS:
                if s in parent_parts:
                    self.by_split[s].setdefault(p.stem, p)
                    break
            self.any.setdefault(p.stem, p)

    def find(self, split: str, image_id: str) -> Path | None:
        stem = Path(image_id).stem if Path(image_id).suffix.lower() in IMG_EXT else image_id
        return self.by_split.get(split, {}).get(stem) or self.any.get(stem)


@lru_cache(maxsize=4)
def get_index(root: str) -> ImageIndex:
    return ImageIndex(Path(root))


def find_csv(root: Path, name: str) -> Path | None:
    """Acha o CSV de um split ('rotulos_5', 'val', 'test_public', ...)."""
    root = Path(root)
    names = [f"{name}.csv"]
    if not name.startswith("rotulos"):
        names += [f"{name}_labels.csv", f"labels_{name}.csv", f"{name}_rotulos.csv",
                  f"rotulos_{name}.csv"]
    for n in names:
        hits = sorted(root.rglob(n), key=lambda q: len(q.parts))
        if hits:
            return hits[0]
    for n in ("labels.csv", "rotulos.csv"):  # CSV dentro da pasta do split
        if (root / name / n).exists():
            return root / name / n
    return None


# --------------------------------------------------------------------------- #
# Itens (id, caminho, rótulo)
# --------------------------------------------------------------------------- #
@dataclass(frozen=True)
class Item:
    image_id: str  # exatamente como aparece no CSV (ou o stem do arquivo, no train)
    path: Path
    label: int     # -1 = sem rótulo


def _pick(columns, preferred) -> str | None:
    low = {str(c).strip().lower(): c for c in columns}
    return next((low[p] for p in preferred if p in low), None)


def _read_labeled_csv(csv: Path, classes: list[str], index: ImageIndex, split_dir: str) -> list[Item]:
    df = pd.read_csv(csv, dtype=str, keep_default_na=False)
    id_col = _pick(df.columns, ID_COLS) or df.columns[0]
    lab_col = _pick(df.columns, LABEL_COLS) or next(c for c in df.columns if c != id_col)
    name2idx = {n.strip().lower(): i for i, n in enumerate(classes)}
    items, missing, unknown = [], [], set()
    for image_id, lab in zip(df[id_col].str.strip(), df[lab_col].str.strip()):
        if lab.isdigit() and int(lab) < len(classes):
            y = int(lab)
        elif lab.lower() in name2idx:
            y = name2idx[lab.lower()]
        else:
            unknown.add(lab)
            continue
        path = index.find(split_dir, image_id)
        if path is None:
            missing.append(image_id)
            continue
        items.append(Item(image_id, path, y))
    if unknown:
        raise ValueError(f"{csv.name}: rótulos que não estão em classes.csv: {sorted(unknown)[:10]}")
    if missing:
        raise FileNotFoundError(
            f"{csv.name}: {len(missing)} imagens do CSV não foram achadas em disco "
            f"(ex.: {missing[:3]}). Colunas usadas: id='{id_col}', rótulo='{lab_col}'."
        )
    return items


def load_split(root: str | Path, name: str, classes: list[str] | None = None) -> list[Item]:
    """Carrega um split. `name` ∈ {train, val, test_public, rotulos_1, rotulos_5, rotulos_20}.

    `train` vem sem rótulo (label = -1): são todas as imagens da pasta train.
    """
    root = Path(root)
    classes = classes or read_classes(root)
    index = get_index(str(root))
    if name == "train":
        if not index.by_split["train"]:
            raise FileNotFoundError(f"Nenhuma imagem em uma pasta 'train' dentro de {root}.")
        return [Item(p.stem, p, -1) for p in sorted(index.by_split["train"].values())]
    csv = find_csv(root, name)
    if csv is None:
        raise FileNotFoundError(
            f"Não achei o CSV de '{name}' em {root}. Rode scripts/inspect_data.py "
            "para ver o que o pacote contém e ajuste find_csv() se preciso."
        )
    split_dir = "train" if name.startswith("rotulos") else name
    return _read_labeled_csv(csv, classes, index, split_dir)


def labeled_budget(root, budget: int, classes=None) -> list[Item]:
    return load_split(root, f"rotulos_{budget}", classes)


def ids_have_extension(items: list[Item]) -> bool:
    """Os image_id dos CSVs trazem extensão (.jpg)? predict.py segue o mesmo formato."""
    return any(Path(i.image_id).suffix.lower() in IMG_EXT for i in items)


def class_counts(items: list[Item], num_classes: int) -> np.ndarray:
    return np.bincount([i.label for i in items if i.label >= 0], minlength=num_classes)


def check_splits(train: list[Item], labeled: list[Item], val: list[Item], test: list[Item]) -> None:
    """Garante: rótulos ⊂ train, e train/val/test_public sem imagens em comum."""
    s = lambda items: {Path(i.image_id).stem for i in items}  # noqa: E731
    tr, lab, va, te = s(train), s(labeled), s(val), s(test)
    problems = []
    if not lab <= tr:
        problems.append(f"{len(lab - tr)} imagens rotuladas não estão no train (ex.: {sorted(lab - tr)[:3]})")
    for (na, a), (nb, b) in [(("train", tr), ("val", va)), (("train", tr), ("test_public", te)),
                              (("val", va), ("test_public", te))]:
        if a & b:
            problems.append(f"{len(a & b)} imagens em '{na}' e '{nb}' ao mesmo tempo (ex.: {sorted(a & b)[:3]})")
    if problems:
        raise AssertionError("Problema de split:\n  " + "\n  ".join(problems))


# --------------------------------------------------------------------------- #
# Dataset e transformações
# --------------------------------------------------------------------------- #
def load_image(path: Path, size: int) -> Image.Image:
    """Abre, converte para RGB e redimensiona para size×size (sem recorte).

    O recorte central cortaria animais pequenos nas bordas; esticar mantém todo o
    campo de visão. O custo é uma leve distorção de proporção, igual no treino e
    no teste.
    """
    try:
        img = Image.open(path).convert("RGB")
        return img.resize((size, size), Image.Resampling.BICUBIC)
    except (OSError, ValueError) as e:
        warnings.warn(f"Falha ao ler {path}: {e}. Usando imagem cinza no lugar.")
        return Image.new("RGB", (size, size), (124, 116, 104))


def train_transform(img_size: int = 224, scale=(0.5, 1.0)) -> T.Compose:
    """Aumentação do treino supervisionado. `scale` baixo demais corta o animal fora."""
    return T.Compose([
        T.RandomResizedCrop(img_size, scale=scale, ratio=(3 / 4, 4 / 3),
                            interpolation=T.InterpolationMode.BICUBIC, antialias=True),
        T.RandomHorizontalFlip(),
        T.ColorJitter(0.3, 0.3, 0.2),
        T.RandomGrayscale(0.1),  # fotos noturnas (infravermelho) são sem cor
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def eval_transform() -> T.Compose:
    return T.Compose([T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


class PA3Dataset(Dataset):
    """Imagens pré-carregadas na RAM em size×size (rápido para treinar várias vezes)."""

    def __init__(self, items: list[Item], size: int, transform=None, preload: bool = True,
                 workers: int = 8):
        self.items, self.size, self.transform = list(items), size, transform
        self.images = None
        if preload:
            with ThreadPoolExecutor(max_workers=workers) as ex:
                self.images = list(ex.map(lambda it: load_image(it.path, size), self.items))

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, i: int):
        img = self.images[i] if self.images is not None else load_image(self.items[i].path, self.size)
        if self.transform is not None:
            img = self.transform(img)
        return img, self.items[i].label

    @property
    def labels(self) -> torch.Tensor:
        return torch.tensor([it.label for it in self.items], dtype=torch.long)


def make_train_ds(items, img_size: int = 224, load_size: int = 256, scale=(0.5, 1.0)) -> PA3Dataset:
    """Carrega em load_size (>= img_size) para o RandomResizedCrop ter folga de resolução."""
    return PA3Dataset(items, load_size, train_transform(img_size, scale))


def make_eval_ds(items, img_size: int = 224, preload: bool = True) -> PA3Dataset:
    return PA3Dataset(items, img_size, eval_transform(), preload=preload)


def make_loader(ds: Dataset, batch_size: int, shuffle: bool, seed: int = 0, num_workers: int = 0) -> DataLoader:
    g = torch.Generator().manual_seed(seed)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle, generator=g, num_workers=num_workers,
        drop_last=shuffle and len(ds) > batch_size,  # BatchNorm não aceita batch de 1
        pin_memory=torch.cuda.is_available(), persistent_workers=num_workers > 0,
    )

Writing pa3/data.py


In [9]:
import sys
sys.path.insert(0, "/content/PA3_Deep_Learning")
from pa3.data import find_root, read_classes, load_split, check_splits, ids_have_extension

root = find_root("/content/data")
classes = read_classes(root)
print(len(classes), "classes; primeiras:", classes[:3])

s = {n: load_split(root, n, classes) for n in
     ["train", "rotulos_1", "rotulos_5", "rotulos_20", "val", "test_public"]}
print({n: len(v) for n, v in s.items()})

check_splits(s["train"], s["rotulos_5"], s["val"], s["test_public"])
print("sem vazamento; ids com extensão?", ids_have_extension(s["rotulos_5"]))

21 classes; primeiras: ['american_black_bear', 'american_crow', 'bird']
{'train': 6505, 'rotulos_1': 21, 'rotulos_5': 105, 'rotulos_20': 420, 'val': 202, 'test_public': 811}
sem vazamento; ids com extensão? False


In [10]:
%%writefile pa3/models.py
"""Modelos do PA3: CNN pequena (do zero) e ResNet-18 (ImageNet), mais checkpoints."""
from __future__ import annotations

from pathlib import Path

import torch
import torch.nn as nn
import torchvision

from . import MAX_PARAMS


class SmallCNN(nn.Module):
    """CNN de 5 blocos (2 convs + BN + ReLU + maxpool), ~2,4 M de parâmetros.

    Serve de piso "do zero" (Parte 0a). O classificador se chama `fc`, como na ResNet,
    para o treino tratar cabeça e corpo igual nos dois modelos.
    """

    def __init__(self, num_classes: int, width: int = 32, dropout: float = 0.3):
        super().__init__()
        chans = [width, width * 2, width * 4, width * 8, width * 8]
        layers, c_in = [], 3
        for c in chans:
            layers += [
                nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(chans[-1], num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.features(x)).flatten(1)
        return self.fc(self.drop(x))


ARCHS = ("small_cnn", "resnet18")


def build_model(arch: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    if arch == "small_cnn":
        return SmallCNN(num_classes)
    if arch == "resnet18":
        weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        model = torchvision.models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model
    raise ValueError(f"arch desconhecida: {arch}. Opções: {ARCHS}")


def count_params(model: nn.Module) -> int:
    """Contagem exatamente como no enunciado: sum(p.numel() for p in model.parameters())."""
    return sum(p.numel() for p in model.parameters())


def check_param_budget(model: nn.Module) -> int:
    n = count_params(model)
    if n > MAX_PARAMS:
        raise ValueError(f"Modelo com {n:,} parâmetros passa do limite de {MAX_PARAMS:,}.")
    return n


# --------------------------------------------------------------------------- #
# Checkpoints: tudo que predict.py precisa para rodar sozinho, sem internet.
# --------------------------------------------------------------------------- #
def save_checkpoint(path: str | Path, model: nn.Module, arch: str, class_names: list[str],
                    img_size: int, id_has_ext: bool, extra: dict | None = None) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "arch": arch,
        "class_names": list(class_names),
        "img_size": int(img_size),
        "id_has_ext": bool(id_has_ext),
        "extra": extra or {},
        "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    }, path)


def load_checkpoint(path: str | Path, map_location="cpu") -> tuple[nn.Module, dict]:
    ckpt = torch.load(path, map_location=map_location, weights_only=True)
    model = build_model(ckpt["arch"], len(ckpt["class_names"]), pretrained=False)  # sem baixar nada
    model.load_state_dict(ckpt["state_dict"])
    meta = {k: v for k, v in ckpt.items() if k != "state_dict"}
    return model.eval(), meta

Writing pa3/models.py


In [11]:
import importlib, pa3.models
importlib.reload(pa3.models)
from pa3.models import build_model, count_params, check_param_budget

for arch, pre in [("small_cnn", False), ("resnet18", True)]:
    m = build_model(arch, 21, pretrained=pre)
    print(f"{arch:10s} pretrained={pre!s:5}  {count_params(m):,} parâmetros  (dentro do limite: {check_param_budget(m) <= 12_000_000})")

small_cnn  pretrained=False  2,359,285 parâmetros  (dentro do limite: True)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 107MB/s]


resnet18   pretrained=True   11,187,285 parâmetros  (dentro do limite: True)


In [12]:
%%writefile pa3/utils.py
"""Utilitários: semente, dispositivo, JSON e tabela da curva de orçamento."""
from __future__ import annotations

import json
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch

CURVA_COLS = ["metodo", "orcamento", "seed", "f1_macro", "acuracia"]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device(name: str = "auto") -> torch.device:
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def git_commit() -> str:
    """Hash curto do commit atual, para amarrar cada resultado ao código que o gerou."""
    try:
        out = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True,
                             timeout=5, cwd=Path(__file__).resolve().parent)
        return out.stdout.strip() or "sem-git"
    except (OSError, subprocess.SubprocessError):
        return "sem-git"


def save_json(obj, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=float))


def append_curve(csv_path: str | Path, metodo: str, orcamento: int, seed: int,
                 f1_macro: float, acuracia: float) -> None:
    """Acrescenta/atualiza uma linha de curva_orcamento.csv (chave: metodo, orcamento, seed)."""
    csv_path = Path(csv_path)
    row = pd.DataFrame([[metodo, orcamento, seed, round(f1_macro, 4), round(acuracia, 4)]], columns=CURVA_COLS)
    if csv_path.exists():
        old = pd.read_csv(csv_path)
        keep = ~((old.metodo == metodo) & (old.orcamento == orcamento) & (old.seed == seed))
        row = pd.concat([old[keep], row], ignore_index=True)
    row = row.sort_values(["metodo", "orcamento", "seed"]).reset_index(drop=True)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    row.to_csv(csv_path, index=False)

Writing pa3/utils.py


In [ ]:
%%writefile pa3/metrics.py
"""Métricas (F1 macro é a principal do PA) e gráficos da Parte 0."""
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score


def summarize(y_true, y_pred, num_classes: int) -> dict:
    """F1 macro sobre as K classes (classe sem acerto conta 0, não some da média)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    labels = list(range(num_classes))
    return {
        "f1_macro": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1_per_class": f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0).tolist(),
        "support": np.bincount(y_true, minlength=num_classes).tolist(),
        "confusion": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def memorization_summary(history: list[dict], train_acc_thr: float = 0.95) -> dict:
    """Responde 'em que época começa a decorar?' a partir da curva.

    - epoch_min_val_loss: depois dela a perda de validação sobe enquanto a de treino cai.
    - epoch_train_acc_thr: primeira época em que a acurácia (com aumentação) passa do limiar.
    """
    val_loss = [h["val_loss"] for h in history]
    hit = next((h["epoch"] for h in history if h["train_acc"] >= train_acc_thr), None)
    return {
        "epoch_min_val_loss": history[int(np.argmin(val_loss))]["epoch"],
        "epoch_best_val_f1": history[int(np.argmax([h["val_f1"] for h in history]))]["epoch"],
        "epoch_train_acc_thr": hit,
        "train_acc_thr": train_acc_thr,
    }


def plot_history(history: list[dict], path: str | Path, title: str = "", best_epoch: int | None = None) -> None:
    ep = [h["epoch"] for h in history]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(ep, [h["train_loss"] for h in history], label="treino")
    ax[0].plot(ep, [h["val_loss"] for h in history], label="validação")
    ax[0].set(xlabel="época", ylabel="perda (cross-entropy)", title="Perda")
    ax[1].plot(ep, [h["train_acc"] for h in history], label="acurácia treino")
    ax[1].plot(ep, [h["val_acc"] for h in history], label="acurácia val")
    ax[1].plot(ep, [h["val_f1"] for h in history], label="F1 macro val", linestyle="--")
    ax[1].set(xlabel="época", ylabel="valor", title="Acurácia e F1", ylim=(0, 1.02))
    for a in ax:
        if best_epoch is not None:
            a.axvline(best_epoch, color="gray", linestyle=":", label=f"época escolhida ({best_epoch})")
        a.xaxis.set_major_locator(MaxNLocator(integer=True))
        a.grid(alpha=0.3)
        a.legend()
    fig.suptitle(title)
    fig.tight_layout()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_confusion(cm, class_names: list[str], path: str | Path, title: str = "") -> None:
    """Matriz de confusão normalizada por linha (recall), com a contagem em cada célula."""
    cm = np.asarray(cm, dtype=float)
    norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    k = len(class_names)
    fig, ax = plt.subplots(figsize=(max(6, 0.55 * k + 3), max(5, 0.55 * k + 2)))
    im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
    ax.set(xticks=range(k), yticks=range(k), xlabel="previsto", ylabel="verdadeiro", title=title)
    ax.set_xticklabels(class_names, rotation=90, fontsize=7)
    ax.set_yticklabels(class_names, fontsize=7)
    for i in range(k):
        for j in range(k):
            if cm[i, j] > 0:
                ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=6,
                        color="white" if norm[i, j] > 0.5 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, label="fração da classe verdadeira")
    fig.tight_layout()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.close(fig)